# 01 — Sales Analysis
## Project Caravela: Olist E-Commerce Analytics

**Metrics covered:** 1 (Monthly GMV), 2 (Top products), 6 (Payment distribution), 7 (AOV trend), 8 (Cancellation rate)

**Observation window:** Jan 2017 – Aug 2018 (excludes 2016 ramp-up and 2018-09/10 data cut artefacts — see `00_eda.ipynb` §5). 2016 had only 312 total orders across Nov–Dec; 2018-09 and 2018-10 have negligible orders in `fct_sales` — incomplete months that would distort trend analysis.

**Export:** `data/sales_orders.parquet` — order-item granularity (~112k rows)

**Source tables:** `fct_sales`, `fct_payments`, `dim_products`, `dim_date`, `dim_customers`

**Key definitions:**
- `total_sale_amount` = `price` + `freight_value` (per item). This is the customer-facing total, not net revenue. Freight averages ~14.2% of GMV.
- `primary_payment_type` / `primary_payment_installments` use `payment_sequential = 1` — the first-listed payment method per order. ~3% of orders split payment across multiple methods (e.g., voucher + credit card); this approximation captures the dominant method only. Aggregate payment values from `fct_payments` remain complete.

> **Currency:** All monetary values in R$ (Brazilian Real). R$15.8M total GMV ≈ USD 4.3M at 2018 average rates (~R$3.65/USD).

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv(Path("..") / ".env")

sys.path.insert(0, str(Path().resolve()))
from utils import (
    REGION_MAP, STATUS_COLOURS, add_region,
    lorenz_curve, gini_coefficient, hhi, concentration_summary,
)

project_id = os.environ["GCP_PROJECT_ID"]
dataset = os.environ.get("BIGQUERY_ANALYTICS_DATASET", "olist_analytics")
client = bigquery.Client(project=project_id)

def run_query(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

print(f"Connected to project: {project_id}  |  Dataset: {dataset}")

Connected to project: praxis-paratext-488407-i1  |  Dataset: olist_analytics


## Data Extraction

Single query joining `fct_sales` with dimension tables and primary payment info (`payment_sequential = 1`). Filtered to the Jan 2017 – Aug 2018 observation window.

**Important:** `fct_sales` is at **order-item granularity** — one row per item in an order. A 3-item order produces 3 rows. All order-level metrics (order count, cancellation rate, payment distribution) must use `COUNT(DISTINCT order_id)` to avoid inflation. Item-level metrics (GMV, revenue by category) can sum directly.

In [2]:
df = run_query(f"""
    SELECT
        s.order_id,
        s.order_item_id,
        s.product_id,
        p.product_category_name_english,
        s.date_key,
        d.year,
        d.month,
        s.order_status,
        s.total_sale_amount,
        s.price,
        s.freight_value,
        pay.payment_type       AS primary_payment_type,
        pay.payment_installments AS primary_payment_installments,
        c.customer_state
    FROM `{project_id}.{dataset}.fct_sales` s
    JOIN `{project_id}.{dataset}.dim_products` p ON s.product_id = p.product_id
    JOIN `{project_id}.{dataset}.dim_date` d    ON s.date_key = d.date_key
    JOIN `{project_id}.{dataset}.dim_customers` c ON s.customer_unique_id = c.customer_unique_id
    LEFT JOIN `{project_id}.{dataset}.fct_payments` pay
        ON s.order_id = pay.order_id AND pay.payment_sequential = 1
    WHERE s.date_key >= '2017-01-01' AND s.date_key <= '2018-08-31'
""")

# Convert BQ-specific dtypes to standard pandas types for Parquet compatibility
df["date_key"] = pd.to_datetime(df["date_key"])
df["year"] = df["year"].astype("int64")
df["month"] = df["month"].astype("int64")
df["order_item_id"] = df["order_item_id"].astype("int64")
df["primary_payment_installments"] = df["primary_payment_installments"].astype("Int64")

df = add_region(df, "customer_state")
print(f"Rows: {len(df):,}  |  Distinct orders: {df['order_id'].nunique():,}")
df.head()

/opt/anaconda3/envs/assignment2/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Rows: 112,279  |  Distinct orders: 98,353


,order_id,order_item_id,product_id,product_category_name_english,date_key,year,month,order_status,total_sale_amount,price,freight_value,primary_payment_type,primary_payment_installments,customer_state,customer_region
0,d455a8cb295653b55abda06d434ab492,1,a2ff5a97bf95719e38ea2e3b4105bce8,small_appliances,2017-09-26,2017,9,delivered,916.02,895.0,21.02,credit_card,10,PR,South
1,9dc8d1a6f16f1b89874c29c9d8d30447,1,a2ff5a97bf95719e38ea2e3b4105bce8,small_appliances,2017-10-12,2017,10,delivered,916.02,895.0,21.02,credit_card,4,MG,Southeast
2,7f39ba4c9052be115350065d07583cac,1,a2ff5a97bf95719e38ea2e3b4105bce8,small_appliances,2017-10-18,2017,10,delivered,916.02,895.0,21.02,credit_card,8,MG,Southeast
3,3317db84e285d6115f358daa9351a4ba,1,08574b074924071f4e201e151b152b4e,garden_tools,2017-04-02,2017,4,delivered,101.13,69.9,31.23,credit_card,2,MG,Southeast
4,6f4eebb5dd7d532cd45474ce133dc60a,1,08574b074924071f4e201e151b152b4e,garden_tools,2017-03-31,2017,3,delivered,101.13,69.9,31.23,credit_card,2,MG,Southeast


## Metric 1 — Monthly GMV & Order Volume

Two stacked panels: area chart for GMV and bar chart for order count. No dual-axis to avoid misleading correlation between scales.

In [3]:
monthly = (
    df.groupby(["year", "month"])
    .agg(gmv=("total_sale_amount", "sum"), orders=("order_id", "nunique"))
    .reset_index()
)
monthly["period"] = monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
monthly = monthly.sort_values("period")

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Monthly GMV (R$)", "Monthly Order Count"))

fig.add_trace(go.Scatter(
    x=monthly["period"], y=monthly["gmv"], fill="tozeroy",
    name="GMV", line=dict(color="#27ae60"),
), row=1, col=1)

fig.add_trace(go.Bar(
    x=monthly["period"], y=monthly["orders"],
    name="Orders", marker_color="#3498db",
), row=2, col=1)

fig.update_layout(height=500, showlegend=False, template="plotly_white")
fig.update_yaxes(title_text="R$", row=1, col=1)
fig.update_yaxes(title_text="Orders", row=2, col=1)
fig.show()

print(f"Total GMV: R${monthly['gmv'].sum():,.2f}")
print(f"Total orders: {monthly['orders'].sum():,}")
print(f"Peak month: {monthly.loc[monthly['orders'].idxmax(), 'period']} "
      f"({monthly['orders'].max():,} orders)")

Total GMV: R$15,786,203.57
Total orders: 98,353
Peak month: 2017-11 (7,451 orders)


**Growth trajectory:** H1 2017 → H2 2017 shows ~103% GMV growth; H2 2017 → H1 2018 slows to ~38%. The marketplace is approaching saturation in its current customer acquisition model — volume growth is decelerating even as absolute numbers remain strong.

## Metric 2 — Top-Selling Products by Revenue

Ranked by `total_sale_amount` (not units). Horizontal bar for ranking clarity and treemap for proportional view.

In [4]:
top_cats = (
    df.groupby("product_category_name_english")["total_sale_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
    .rename(columns={"total_sale_amount": "revenue"})
)

# Horizontal bar — sorted descending
fig1 = px.bar(
    top_cats.sort_values("revenue"),
    x="revenue", y="product_category_name_english",
    orientation="h",
    title="Top 15 Product Categories by Revenue",
    labels={"revenue": "Revenue (R$)", "product_category_name_english": "Category"},
    template="plotly_white",
)
fig1.update_traces(marker_color="#e67e22")
fig1.show()

# Treemap — proportional view
fig2 = px.treemap(
    top_cats, path=["product_category_name_english"], values="revenue",
    title="Revenue Share — Top 15 Categories",
    color="revenue", color_continuous_scale="YlOrRd",
)
fig2.update_layout(height=450)
fig2.show()

print(f"Top 15 categories account for R${top_cats['revenue'].sum():,.2f} "
      f"({100*top_cats['revenue'].sum()/df['total_sale_amount'].sum():.1f}% of total GMV)")

Top 15 categories account for R$12,062,003.39 (76.4% of total GMV)


### Revenue Concentration: How Diversified Is the Product Mix?

**Why this matters:** The top-15 chart above shows *which* categories lead, but doesn't quantify *how far* the revenue gap extends. If the top 3 categories disappeared tomorrow (supplier issues, regulatory change, trend shift), what fraction of revenue would Olist lose?

**Question it answers:** *"Is Olist's revenue resilient across many product categories, or dangerously dependent on a few verticals?"*

The **Lorenz curve** here plots cumulative category share (sorted from smallest to largest revenue) against cumulative share of total revenue. A steep curve means a few categories carry most of the weight.

For product mix analysis, the **Gini coefficient** tells us whether revenue is spread broadly across categories or concentrated in a few. Combined with **CR4** (the share held by the top 4 categories), this gives an executive-level risk metric.

In [5]:
# --- Category Revenue Concentration: Lorenz Curve & Gini ---
cat_revenue = (
    df.groupby("product_category_name_english")["total_sale_amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"total_sale_amount": "revenue"})
)

cat_values = cat_revenue["revenue"].values

# Lorenz curve
lx, ly = lorenz_curve(cat_values)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lx, y=ly, mode="lines", name="Lorenz Curve (Categories)",
    line=dict(color="#e67e22", width=2.5),
    fill="tozeroy", fillcolor="rgba(230, 126, 34, 0.12)",
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Perfect Equality",
    line=dict(color="#95a5a6", width=1.5, dash="dash"),
))
fig.update_layout(
    title="Product Category Revenue Lorenz Curve",
    xaxis_title="Cumulative Share of Categories (sorted smallest → largest)",
    yaxis_title="Cumulative Share of Revenue",
    template="plotly_white", height=420,
    legend=dict(x=0.05, y=0.95),
    xaxis=dict(tickformat=".0%"), yaxis=dict(tickformat=".0%"),
)
fig.show()

# Concentration metrics
cat_conc = concentration_summary(cat_values, name="category_revenue")

print("=" * 60)
print("PRODUCT CATEGORY REVENUE CONCENTRATION")
print("=" * 60)
print(f"  Gini Coefficient:   {cat_conc['gini']:.4f}")
print(f"  CR4 (top 4 share):  {cat_conc['cr4']:.1%}")
print(f"  CR10 (top 10 share):{cat_conc['cr10']:.1%}")
print(f"  HHI:                {cat_conc['hhi']:.0f}", end="")
if cat_conc['hhi'] < 1500:
    print("  → Competitive (no category dominance)")
else:
    print("  → Concentrated")
print(f"  Top 20% share:      {cat_conc['top_20pct_share']:.1%}")
print(f"  Categories:         {cat_conc['n_entities']}")
print()

# Interpretation — clearly distinguish Gini (revenue inequality) from HHI (dominance)
print(f"→ Gini of {cat_conc['gini']:.2f} shows revenue is unevenly distributed across")
print("  categories — a small number of categories earn disproportionately more.")
print(f"  However, HHI = {cat_conc['hhi']:.0f} (well below the 1,500 threshold) confirms")
print("  that no single category dominates the marketplace. CR4 = {:.1%} means".format(cat_conc['cr4']))
print(f"  the top 4 categories together hold less than a third of revenue.")
print()
print("  In short: revenue is concentrated in popular categories (Gini effect)")
print("  but the portfolio is diversified enough that Olist isn't over-dependent")
print("  on any one product vertical (HHI conclusion). This is a healthy pattern")
print("  for a general marketplace.")

PRODUCT CATEGORY REVENUE CONCENTRATION
  Gini Coefficient:   0.7134
  CR4 (top 4 share):  32.5%
  CR10 (top 10 share):62.4%
  HHI:                484  → Competitive (no category dominance)
  Top 20% share:      76.4%
  Categories:         74

→ Gini of 0.71 shows revenue is unevenly distributed across
  categories — a small number of categories earn disproportionately more.
  However, HHI = 484 (well below the 1,500 threshold) confirms
  that no single category dominates the marketplace. CR4 = 32.5% means
  the top 4 categories together hold less than a third of revenue.

  In short: revenue is concentrated in popular categories (Gini effect)
  but the portfolio is diversified enough that Olist isn't over-dependent
  on any one product vertical (HHI conclusion). This is a healthy pattern
  for a general marketplace.


## Metric 6 — Payment Method Distribution

Donut chart for payment type share and histogram for credit card installment distribution.

In [6]:
# Payment type share (order-level — use distinct orders to avoid item-granularity inflation)
pay_orders = df.drop_duplicates(subset="order_id")
pay_type = pay_orders["primary_payment_type"].value_counts().reset_index()
pay_type.columns = ["payment_type", "orders"]

fig1 = px.pie(
    pay_type, values="orders", names="payment_type",
    title="Payment Type Distribution (by orders)",
    hole=0.45, template="plotly_white",
    color_discrete_sequence=["#3498db", "#e67e22", "#2ecc71", "#9b59b6"],
)
fig1.show()

# Credit card installment distribution
cc = pay_orders[pay_orders["primary_payment_type"] == "credit_card"]
fig2 = px.histogram(
    cc, x="primary_payment_installments", nbins=12,
    title="Credit Card Installment Distribution",
    labels={"primary_payment_installments": "Installments"},
    template="plotly_white",
)
fig2.update_traces(marker_color="#3498db")
fig2.show()

print(f"Credit card orders: {len(cc):,} ({100*len(cc)/len(pay_orders):.1f}%)")
print(f"Median installments (credit card): {cc['primary_payment_installments'].median():.0f}")
print(f"Mean installments (credit card):   {cc['primary_payment_installments'].mean():.1f}")

Credit card orders: 75,716 (77.0%)
Median installments (credit card): 3
Mean installments (credit card):   3.5


## Metric 7 — Average Order Value (AOV) Trend

Line chart showing AOV by month, plus bar chart comparing AOV across payment types.

In [7]:
# AOV = total GMV / distinct orders (order-level metric)
aov_monthly = (
    df.groupby(["year", "month"])
    .apply(lambda g: pd.Series({
        "gmv": g["total_sale_amount"].sum(),
        "orders": g["order_id"].nunique(),
    }), include_groups=False)
    .reset_index()
)
aov_monthly["aov"] = aov_monthly["gmv"] / aov_monthly["orders"]
aov_monthly["period"] = aov_monthly["year"].astype(str) + "-" + aov_monthly["month"].astype(str).str.zfill(2)
aov_monthly = aov_monthly.sort_values("period")

fig1 = px.line(
    aov_monthly, x="period", y="aov",
    title="Average Order Value (AOV) by Month",
    labels={"aov": "AOV (R$)", "period": "Month"},
    template="plotly_white", markers=True,
)
fig1.update_traces(line_color="#e67e22")
fig1.show()

# AOV by payment type
aov_pay = (
    pay_orders.groupby("primary_payment_type")
    .agg(gmv=("total_sale_amount", "sum"), orders=("order_id", "nunique"))
    .reset_index()
)
aov_pay["aov"] = aov_pay["gmv"] / aov_pay["orders"]
aov_pay = aov_pay.sort_values("aov", ascending=False)

fig2 = px.bar(
    aov_pay, x="primary_payment_type", y="aov",
    title="AOV by Payment Type",
    labels={"aov": "AOV (R$)", "primary_payment_type": "Payment Type"},
    template="plotly_white", text_auto=".2f",
)
fig2.update_traces(marker_color="#9b59b6")
fig2.show()

overall_aov = df["total_sale_amount"].sum() / df["order_id"].nunique()
print(f"Overall AOV: R${overall_aov:,.2f}")

Overall AOV: R$160.51


## Metric 8 — Fulfilment / Cancellation Rate

Line chart tracking cancellation % and unavailability % over time, plus donut showing overall order status mix. Stacked bar excluded because `delivered` (96.8%) would make all other statuses invisible.

In [8]:
# Monthly cancellation and unavailability rates (order-level)
orders_monthly = df.drop_duplicates(subset="order_id")
status_monthly = (
    orders_monthly.groupby(["year", "month", "order_status"])["order_id"]
    .count()
    .reset_index(name="cnt")
)
total_monthly = (
    orders_monthly.groupby(["year", "month"])["order_id"]
    .count()
    .reset_index(name="total")
)
status_monthly = status_monthly.merge(total_monthly, on=["year", "month"])
status_monthly["pct"] = 100 * status_monthly["cnt"] / status_monthly["total"]
status_monthly["period"] = status_monthly["year"].astype(str) + "-" + status_monthly["month"].astype(str).str.zfill(2)

cancel_line = status_monthly[status_monthly["order_status"].isin(["canceled", "unavailable"])]
cancel_line = cancel_line.sort_values("period")

fig1 = px.line(
    cancel_line, x="period", y="pct", color="order_status",
    title="Cancellation & Unavailability Rate Over Time",
    labels={"pct": "% of Orders", "period": "Month"},
    template="plotly_white", markers=True,
    color_discrete_map={"canceled": "#e74c3c", "unavailable": "#c0392b"},
)
fig1.show()

# Overall status donut
status_overall = (
    orders_monthly["order_status"].value_counts().reset_index()
)
status_overall.columns = ["order_status", "orders"]
color_map = {s: STATUS_COLOURS.get(s, "#95a5a6") for s in status_overall["order_status"]}

fig2 = px.pie(
    status_overall, values="orders", names="order_status",
    title="Overall Order Status Distribution",
    hole=0.45, template="plotly_white",
    color="order_status", color_discrete_map=color_map,
)
fig2.show()

total_o = orders_monthly["order_id"].count()
cancel_n = (orders_monthly["order_status"] == "canceled").sum()
unavail_n = (orders_monthly["order_status"] == "unavailable").sum()
print(f"Overall cancellation rate: {100*cancel_n/total_o:.2f}%")
print(f"Overall unavailability rate: {100*unavail_n/total_o:.2f}%")

Overall cancellation rate: 0.46%
Overall unavailability rate: 0.00%


## Additional Analysis: Freight Cost Structure

Freight accounts for 14.2% of total GMV — a significant hidden cost for consumers. Here we examine which product categories have the highest freight-to-price ratio, identifying where shipping costs may suppress conversion.

In [9]:
# Freight-to-price ratio by category (min 100 items for statistical relevance)
freight_cat = (
    df.groupby("product_category_name_english")
    .agg(freight=("freight_value", "sum"), price=("price", "sum"), items=("order_id", "count"))
    .reset_index()
)
freight_cat = freight_cat[freight_cat["items"] >= 100]
freight_cat["freight_ratio"] = freight_cat["freight"] / freight_cat["price"]
freight_cat = freight_cat.sort_values("freight_ratio", ascending=False).head(15)

fig = px.bar(
    freight_cat.sort_values("freight_ratio"),
    x="freight_ratio", y="product_category_name_english", orientation="h",
    title="Freight-to-Price Ratio by Category (top 15, min 100 items)",
    labels={"freight_ratio": "Freight / Price", "product_category_name_english": "Category"},
    template="plotly_white", text_auto=".1%",
)
fig.update_traces(marker_color="#e74c3c")
fig.show()

total_freight = df["freight_value"].sum()
total_gmv = df["total_sale_amount"].sum()
print(f"Total freight: R${total_freight:,.2f} ({100*total_freight/total_gmv:.1f}% of GMV)")
print(f"≈ USD${total_freight/3.5:,.0f} at avg 2017-18 rate")
print(f"\nHighest freight categories: christmas_supplies (36.7%), signaling_and_security (30.3%), food_drink (29.7%)")
print("Furniture categories (living room, office, decor) all exceed 23% — weight/volume drives shipping cost.")

Total freight: R$2,244,490.79 (14.2% of GMV)
≈ USD$641,283 at avg 2017-18 rate

Highest freight categories: christmas_supplies (36.7%), signaling_and_security (30.3%), food_drink (29.7%)
Furniture categories (living room, office, decor) all exceed 23% — weight/volume drives shipping cost.


## Additional Analysis: Black Friday Deep-Dive (Nov 2017)

Nov 2017 had 7,451 orders — 63% above the preceding month and the single largest spike in the dataset. Was this driven by discounts (lower AOV), new customers, or simply more traffic?

In [10]:
# Q4 2017 comparison — Sep through Dec
q4 = df[df["year"] == 2017]
q4 = q4[q4["month"].isin([9, 10, 11, 12])]
q4_summary = (
    q4.groupby("month")
    .agg(
        orders=("order_id", "nunique"),
        gmv=("total_sale_amount", "sum"),
    )
    .reset_index()
)
q4_summary["aov"] = q4_summary["gmv"] / q4_summary["orders"]
q4_summary["month_name"] = q4_summary["month"].map({9: "Sep", 10: "Oct", 11: "Nov (BF)", 12: "Dec"})

fig = px.bar(
    q4_summary, x="month_name", y=["orders", "aov"],
    barmode="group",
    title="Q4 2017: Orders & AOV — Black Friday Effect",
    template="plotly_white",
)
fig.show()

print("Q4 2017 breakdown:")
for _, r in q4_summary.iterrows():
    print(f"  {r['month_name']:>10}: {r['orders']:,} orders, AOV R${r['aov']:.2f}, GMV R${r['gmv']:,.0f}")
print(f"\nBlack Friday insight: AOV DROPPED from R$168 (Oct) to R$158 (Nov) — a 6% decline.")
print("The spike is entirely volume-driven, likely fueled by discounts/promotions.")
print("Dec post-BF hangover: orders dropped 25% but AOV stayed low (R$154) — discount habit persists.")

Q4 2017 breakdown:
         Sep: 4,243 orders, AOV R$169.79, GMV R$720,399
         Oct: 4,568 orders, AOV R$168.41, GMV R$769,312
    Nov (BF): 7,451 orders, AOV R$158.25, GMV R$1,179,144
         Dec: 5,624 orders, AOV R$153.55, GMV R$863,547

Black Friday insight: AOV DROPPED from R$168 (Oct) to R$158 (Nov) — a 6% decline.
The spike is entirely volume-driven, likely fueled by discounts/promotions.
Dec post-BF hangover: orders dropped 25% but AOV stayed low (R$154) — discount habit persists.


## Parquet Export

Exporting the full dataset to `data/sales_orders.parquet` at order-item granularity for dashboard consumption.

In [11]:
output_path = Path("..") / "data" / "sales_orders.parquet"
df.to_parquet(output_path, index=False)

# Verify
verify = pd.read_parquet(output_path)
print(f"Exported: {output_path}")
print(f"Rows: {len(verify):,}  |  Columns: {list(verify.columns)}")
print(f"Distinct orders: {verify['order_id'].nunique():,}")
verify.dtypes

Exported: ../data/sales_orders.parquet
Rows: 112,279  |  Columns: ['order_id', 'order_item_id', 'product_id', 'product_category_name_english', 'date_key', 'year', 'month', 'order_status', 'total_sale_amount', 'price', 'freight_value', 'primary_payment_type', 'primary_payment_installments', 'customer_state', 'customer_region']
Distinct orders: 98,353


order_id                                 object
order_item_id                             int64
product_id                               object
product_category_name_english            object
date_key                         datetime64[ns]
year                                      int64
month                                     int64
order_status                             object
total_sale_amount                       float64
price                                   float64
freight_value                           float64
primary_payment_type                     object
primary_payment_installments              Int64
customer_state                           object
customer_region                          object
dtype: object

## Key Findings

> **Currency:** R$ (Brazilian Real). R$15.8M total GMV ≈ USD 4.3M at 2018 avg rate (~R$3.65/USD).

1. **Growth is decelerating**: H1→H2 2017 saw ~103% GMV growth; H2 2017→H1 2018 slowed to ~38%. Monthly orders plateau around 6,500–7,200 from Feb–Aug 2018.
2. **Product concentration is moderate (Gini 0.71, HHI 484)**: Top 15 categories account for ~76% of GMV, but CR4 = 32.5% and no single category exceeds 10%. Health & beauty leads (9.1%), followed by watches/gifts (8.2%) and bed/bath/table (7.9%). The high Gini reflects revenue inequality across categories (a few earn much more than the rest), but the low HHI confirms no single category dominates — the portfolio is diversified.
3. **Credit card dominance**: 77% of orders use credit card with a median of 3 installments (mean 3.5). 7.3% of credit card orders use 10+ installments — installment availability enables larger purchases (CC AOV R$152 vs boleto R$127).
4. **Stable AOV**: Overall AOV ~R$160 is relatively flat across months, confirming growth is volume-driven, not ticket-size-driven.
5. **Low cancellation**: Overall cancellation rate 0.46%; unavailability rate near 0%. Fulfilment is strong.
6. **Freight is a hidden 14.2% tax**: R$2.2M freight on R$15.8M GMV. Christmas supplies (36.7%), food/drink (29.7%), and furniture categories (23–26%) have the highest freight-to-price ratios — weight/volume drives shipping cost.
7. **Black Friday is discount-driven**: Nov 2017 AOV *dropped* 6% (R$168→R$158) while volume surged 63%. The spike is entirely volume-driven, with a post-BF hangover in December (orders -25%, AOV stayed low). This suggests promotional pricing, not organic demand growth.